# Entrenamiento de RT-DETR para matrículas (repositorio oficial)

In [2]:
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps

YOLO_DATASET = Path('yolo_dataset')
COCO_DIR = Path('rtdetr_dataset').resolve()
REPO_URL = 'https://github.com/lyuwenyu/RT-DETR.git'
REPO = Path('RT-DETR-oficial/rtdetrv2_pytorch').resolve()
SALIDA = Path('runs_rtdetr').resolve()
NOMBRE = 'rtdetrv2_r18_matriculas'
PESOS_URL = 'https://github.com/lyuwenyu/storage/releases/download/v0.2/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth'
PESOS = Path('weights/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth').resolve()
EPOCAS = 60
BATCH = 8
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LADO_MAXIMO = 1280
Image.MAX_IMAGE_PIXELS = None

assert YOLO_DATASET.exists(), 'Ejecuta antes VC_P4_entrenamiento_yolo.ipynb'

## Repositorio, pesos y dataset

In [3]:
if not (REPO / 'tools' / 'train.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'RT-DETR-oficial'], check=True)
sys.path.insert(0, str(REPO))

PESOS.parent.mkdir(exist_ok=True)
if not PESOS.exists():
    urllib.request.urlretrieve(PESOS_URL, PESOS)

In [3]:
def a_coco(split):
    destino = COCO_DIR / split / 'images'
    destino.mkdir(parents=True, exist_ok=True)
    imagenes, anotaciones = [], []
    for i, ruta in enumerate(sorted((YOLO_DATASET / split / 'images').iterdir()), 1):
        with Image.open(ruta) as original:
            original.draft('RGB', (LADO_MAXIMO, LADO_MAXIMO))
            imagen = ImageOps.exif_transpose(original).convert('RGB')
        imagen.thumbnail((LADO_MAXIMO, LADO_MAXIMO), Image.LANCZOS)
        ancho, alto = imagen.size
        imagen.save(destino / ruta.name, quality=95)
        imagenes.append({'id': i, 'file_name': ruta.name, 'width': ancho, 'height': alto})
        etiqueta = YOLO_DATASET / split / 'labels' / f'{ruta.stem}.txt'
        for linea in etiqueta.read_text().splitlines():
            _, xc, yc, w, h = map(float, linea.split())
            anotaciones.append({
                'id': len(anotaciones) + 1, 'image_id': i, 'category_id': 0,
                'bbox': [(xc - w / 2) * ancho, (yc - h / 2) * alto, w * ancho, h * alto],
                'area': w * ancho * h * alto, 'iscrowd': 0,
            })
    datos = {'images': imagenes, 'annotations': anotaciones, 'categories': [{'id': 0, 'name': 'plate'}]}
    (COCO_DIR / f'instances_{split}.json').write_text(json.dumps(datos))
    print(f'{split:5s}: {len(imagenes):4d} imágenes, {len(anotaciones):4d} matrículas')


for split in ['train', 'val', 'test']:
    a_coco(split)

train: 1672 imágenes, 3646 matrículas
val  :  357 imágenes,  807 matrículas
test :  357 imágenes,  782 matrículas


## Configuración

In [4]:
PLANTILLA = """__include__: ['@@REPO@@/configs/rtdetrv2/rtdetrv2_r18vd_120e_coco.yml']

output_dir: @@SALIDA@@/@@NOMBRE@@

num_classes: 1
remap_mscoco_category: False

epoches: @@EPOCAS@@
use_ema: False
use_amp: True
sync_bn: False
checkpoint_freq: 1000
print_freq: 20

PResNet:
  pretrained: False

lr_warmup_scheduler:
  type: LinearWarmup
  warmup_duration: 100

train_dataloader:
  total_batch_size: @@BATCH@@
  num_workers: @@NUM_WORKERS@@
  dataset:
    img_folder: @@COCO@@/train/images/
    ann_file: @@COCO@@/instances_train.json
    transforms:
      ops:
        - {type: RandomPhotometricDistort, p: 0.5}
        - {type: RandomZoomOut, fill: 0}
        - {type: RandomIoUCrop, p: 0.8}
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: Resize, size: [640, 640]}
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: ConvertPILImage, dtype: 'float32', scale: True}
        - {type: ConvertBoxes, fmt: 'cxcywh', normalize: True}
      policy:
        name: stop_epoch
        epoch: @@POLITICA@@
        ops: ['RandomPhotometricDistort', 'RandomZoomOut', 'RandomIoUCrop']

val_dataloader:
  total_batch_size: @@BATCH_VAL@@
  num_workers: @@NUM_WORKERS@@
  dataset:
    img_folder: @@COCO@@/val/images/
    ann_file: @@COCO@@/instances_val.json
"""


def escribir_config():
    SALIDA.mkdir(parents=True, exist_ok=True)
    valores = {
        'REPO': REPO.as_posix(), 'SALIDA': SALIDA.as_posix(), 'NOMBRE': NOMBRE, 'COCO': COCO_DIR.as_posix(),
        'EPOCAS': EPOCAS, 'BATCH': BATCH, 'BATCH_VAL': BATCH * 2, 'NUM_WORKERS': NUM_WORKERS,
        'POLITICA': int(EPOCAS * 0.85),
    }
    texto = PLANTILLA
    for clave, valor in valores.items():
        texto = texto.replace(f'@@{clave}@@', str(valor))
    ruta = SALIDA / f'{NOMBRE}.yml'
    ruta.write_text(texto, encoding='utf-8')
    return ruta


CONFIG = escribir_config()
print(CONFIG)

C:\Users\jdmas\OneDrive\Documentos\Ingeniería Informática\4º Año\1º Semestre\Visión por Computador\Práctica\VC-Practicas\P4\runs_rtdetr\rtdetrv2_r18_matriculas.yml


## Entrenamiento

In [5]:
from collections import deque


def ejecutar(comando):
    ultimas = deque(maxlen=40)
    proceso = subprocess.Popen(
        comando, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'},
    )
    try:
        for linea in proceso.stdout:
            print(linea, end='')
            ultimas.append(linea)
    except BaseException:
        proceso.kill()
        raise
    if proceso.wait() != 0:
        raise RuntimeError(f'El proceso terminó con código {proceso.returncode}\n' + ''.join(ultimas))


inicio = time.time()
ejecutar([sys.executable, 'tools/train.py', '-c', str(CONFIG), '-t', str(PESOS), '--seed', '0', '-d', DEVICE.type])
print(f'Tiempo de entrenamiento: {(time.time() - inicio) / 60:.1f} min')

Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': 'C:\\Users\\jdmas\\OneDrive\\Documentos\\Ingeniería Informática\\4º Año\\1º Semestre\\Visión por Computador\\Práctica\\VC-Practicas\\P4\\weights\\rtdetrv2_r18vd_120e_coco_rerun_48.1.pth', 'epoches': 60, 'last_epoch': -1, 'use_amp': False, 'use_ema': False, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': False, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': 0, 'print_freq': 20, 'checkpoint_freq': 1000, 'output_dir': 'C:/Users/jdmas/OneDriv

## Análisis del entrenamiento

In [5]:
registros = [json.loads(linea) for linea in (SALIDA / NOMBRE / 'log.txt').read_text().splitlines()]
curvas = pd.DataFrame({
    'epoca': [x['epoch'] + 1 for x in registros],
    'perdida': [x['train_loss'] for x in registros],
    'AP50': [x['test_coco_eval_bbox'][1] for x in registros],
    'AP50-95': [x['test_coco_eval_bbox'][0] for x in registros],
})

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(curvas['epoca'], curvas['perdida'])
ejes[0].set_xlabel('Época')
ejes[0].set_ylabel('Pérdida de entrenamiento')
ejes[0].set_title('Evolución de la pérdida')
ejes[1].plot(curvas['epoca'], curvas['AP50'], label='AP50')
ejes[1].plot(curvas['epoca'], curvas['AP50-95'], label='AP50-95')
ejes[1].set_xlabel('Época')
ejes[1].set_ylabel('AP en validación')
ejes[1].set_title('Evolución de las métricas')
ejes[1].legend()
plt.tight_layout()
Path('assets').mkdir(exist_ok=True)
plt.savefig('assets/curvas_rtdetr.png', dpi=150)
plt.show()

curvas.tail()

: 

## Comparación con YOLO en el conjunto de test

In [ ]:
from faster_coco_eval import COCO, COCOeval_faster
from src.core import YAMLConfig
from src.core.yaml_utils import load_config
from ultralytics import YOLO


def cargar_rtdetr(pesos):
    load_config.__defaults__[0].clear()
    cfg = YAMLConfig(str(CONFIG))
    checkpoint = torch.load(pesos, map_location='cpu', weights_only=False)
    cfg.model.load_state_dict(checkpoint['ema']['module'] if 'ema' in checkpoint else checkpoint['model'])
    return cfg.model.deploy().to(DEVICE), cfg.postprocessor.deploy().to(DEVICE)


modelo_rtdetr, postprocesador = cargar_rtdetr(SALIDA / NOMBRE / 'best.pth')
modelo_yolo = YOLO('matriculas_best.pt')
YOLO_DEVICE = 0 if DEVICE.type == 'cuda' else 'cpu'

gt = COCO(str(COCO_DIR / 'instances_test.json'))
test = [(i['id'], cv2.imread(str(COCO_DIR / 'test' / 'images' / i['file_name']))) for i in gt.dataset['images']]
print('Imágenes de test:', len(test))

In [ ]:
def predecir_yolo(im):
    resultado = modelo_yolo.predict(im, conf=0.001, iou=0.7, imgsz=640, device=YOLO_DEVICE, verbose=False)[0]
    return resultado.boxes.xyxy.cpu().numpy(), resultado.boxes.conf.cpu().numpy()


@torch.no_grad()
def predecir_rtdetr(im):
    alto, ancho = im.shape[:2]
    entrada = Image.fromarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).resize((640, 640), Image.BILINEAR)
    entrada = torch.from_numpy(np.array(entrada)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)
    _, cajas, puntuaciones = postprocesador(modelo_rtdetr(entrada), torch.tensor([[ancho, alto]], device=DEVICE))
    return cajas[0].cpu().numpy(), puntuaciones[0].cpu().numpy()


def sincronizar():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def medir(predecir):
    for _, im in test[:3]:
        predecir(im)
    resultados, tiempos = [], []
    for id_imagen, im in test:
        sincronizar()
        t0 = time.perf_counter()
        cajas, puntuaciones = predecir(im)
        sincronizar()
        tiempos.append((time.perf_counter() - t0) * 1000)
        for (x1, y1, x2, y2), s in zip(cajas, puntuaciones):
            resultados.append({'image_id': id_imagen, 'category_id': 0,
                               'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s)})
    evaluador = COCOeval_faster(gt, gt.loadRes(resultados), 'bbox')
    evaluador.evaluate()
    evaluador.accumulate()
    evaluador.summarize()
    return {'AP50': evaluador.stats[1], 'AP50-95': evaluador.stats[0], 'AR100': evaluador.stats[8],
            'ms_por_imagen': float(np.mean(tiempos))}


parametros = lambda m: sum(p.numel() for p in m.parameters()) / 1e6

filas = [
    {'modelo': 'YOLO (Ultralytics)', 'parametros_M': parametros(modelo_yolo.model), **medir(predecir_yolo)},
    {'modelo': 'RT-DETRv2-R18 (oficial)', 'parametros_M': parametros(modelo_rtdetr), **medir(predecir_rtdetr)},
]
tabla = pd.DataFrame(filas).round(3)
tabla.to_csv('comparativa_deteccion.csv', index=False)
tabla

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(tabla))
ancho = 0.35
b1 = ejes[0].bar(x - ancho / 2, tabla['AP50'], ancho, label='AP50')
b2 = ejes[0].bar(x + ancho / 2, tabla['AP50-95'], ancho, label='AP50-95')
for barra in list(b1) + list(b2):
    ejes[0].annotate(f'{barra.get_height():.3f}', (barra.get_x() + barra.get_width() / 2, barra.get_height()),
                     ha='center', va='bottom', fontsize=9)
ejes[0].set_xticks(x)
ejes[0].set_xticklabels(tabla['modelo'])
ejes[0].set_ylim(0, 1.1)
ejes[0].set_title('Precisión en test')
ejes[0].legend()

b3 = ejes[1].bar(x, tabla['ms_por_imagen'], color=['#4c78a8', '#f58518'])
for barra in b3:
    ejes[1].annotate(f'{barra.get_height():.1f} ms', (barra.get_x() + barra.get_width() / 2, barra.get_height()),
                     ha='center', va='bottom', fontsize=9)
ejes[1].set_xticks(x)
ejes[1].set_xticklabels(tabla['modelo'])
ejes[1].set_ylabel('ms por imagen')
ejes[1].set_title('Tiempo de inferencia')

plt.tight_layout()
Path('assets').mkdir(exist_ok=True)
plt.savefig('assets/comparativa_deteccion.png', dpi=150)
plt.show()

## Ejemplos

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
for eje, (_, im) in zip(ejes.ravel(), test[:6]):
    salida = im.copy()
    for predecir, color in [(predecir_yolo, (0, 200, 0)), (predecir_rtdetr, (0, 0, 255))]:
        cajas, puntuaciones = predecir(im)
        for (x1, y1, x2, y2), s in zip(cajas, puntuaciones):
            if s >= 0.5:
                cv2.rectangle(salida, (int(x1), int(y1)), (int(x2), int(y2)), color, max(2, im.shape[1] // 300))
    eje.imshow(cv2.cvtColor(salida, cv2.COLOR_BGR2RGB))
    eje.axis('off')
plt.tight_layout()
plt.show()